In [ ]:
# !pip uninstall -y numpy scipy tensorflow tensorflow-gpu
# !pip install numpy==1.23.5
# !pip install scipy==1.10.1
# !pip install tensorflow==2.12.0
# !pip install gensim==4.3.1
# !pip install conllu

# Part-of-Speech Tagging with Bidirectional Stacked RNN

This notebook implements a Part-of-Speech (POS) tagger using a Bidirectional Stacked RNN (with GRU/LSTM cells)
for one of the languages from the Universal Dependencies treebanks.

## 1. Introduction

Part-of-speech tagging is the process of assigning a part-of-speech tag (such as noun, verb, adjective, etc.)
to each word in a text. In this implementation, we use a bidirectional stacked RNN to capture contextual information
from both directions. We use data from the Universal Dependencies treebanks and pre-trained word embeddings as features.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import os
import urllib.request
import zipfile
import conllu
import gensim.downloader as api
from tqdm import tqdm

# TensorFlow imports
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (Dense, Dropout, Embedding, LSTM, GRU, Bidirectional,
                                   TimeDistributed, Input, Flatten, Layer,
                                   Concatenate, Masking, Lambda)
from tensorflow.keras.utils import to_categorical, pad_sequences
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import register_keras_serializable

# Scikit-learn imports
from sklearn.metrics import precision_recall_curve, auc, precision_score, recall_score, f1_score, confusion_matrix

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)


## 2. Dataset

We'll use the Universal Dependencies treebanks for our POS tagging task.
These treebanks provide annotated text data for multiple languages with part-of-speech tags.


In [ ]:
LANGUAGE = "English"
TREEBANK = "UD_English-EWT"
TREEBANK_URL = "https://github.com/UniversalDependencies/UD_English-EWT/archive/master.zip"

# Constants for model configuration
MAX_SEQUENCE_LENGTH = 50  # Maximum sentence length for RNN
EMBEDDING_DIM = 100  # Dimension of word embeddings
CHAR_EMBEDDING_DIM = 50  # Dimension of character embeddings
MAX_WORD_LENGTH = 20  # Maximum word length for character-level features
BATCH_SIZE = 32
EPOCHS = 50
RNN_UNITS = [128, 64]  # Units for each RNN layer
DROPOUT_RATE = 0.3
RECURRENT_DROPOUT = 0.2
USE_CHAR_EMBEDDINGS = True  # Whether to use character-level embeddings
RNN_TYPE = 'LSTM'  # 'LSTM' or 'GRU'

# Function to download and extract the treebank
def download_treebank(url, treebank_name):
    zip_path = f"{treebank_name}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {treebank_name}...")
        urllib.request.urlretrieve(url, zip_path)

    extract_dir = f"{treebank_name}-data"
    if not os.path.exists(extract_dir):
        print(f"Extracting {treebank_name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

    return extract_dir

# Function to load and parse the CoNLL-U files
def load_conllu_data(treebank_dir, treebank_name):
    # Find the CoNLL-U files for train, dev, and test
    base_dir = os.path.join(treebank_dir, f"{treebank_name}-master")
    train_file = None
    dev_file = None
    test_file = None

    for file in os.listdir(base_dir):
        if file.endswith(".conllu"):
            if "train" in file:
                train_file = os.path.join(base_dir, file)
            elif "dev" in file:
                dev_file = os.path.join(base_dir, file)
            elif "test" in file:
                test_file = os.path.join(base_dir, file)

    # Load and parse the CoNLL-U files
    train_data = []
    dev_data = []
    test_data = []

    if train_file:
        with open(train_file, "r", encoding="utf-8") as f:
            train_data = conllu.parse(f.read())

    if dev_file:
        with open(dev_file, "r", encoding="utf-8") as f:
            dev_data = conllu.parse(f.read())

    if test_file:
        with open(test_file, "r", encoding="utf-8") as f:
            test_data = conllu.parse(f.read())

    return train_data, dev_data, test_data

# Function to extract sentences and POS tags from the parsed data
def extract_sentences_and_tags(data):
    sentences = []
    pos_tags = []

    for sentence in data:
        words = []
        tags = []

        for token in sentence:
            # Skip tokens that are not words (e.g., punctuation)
            if token["upos"] != "_":
                words.append(token["form"].lower())
                tags.append(token["upos"])

        if words:  # Only add non-empty sentences
            sentences.append(words)
            pos_tags.append(tags)

    return sentences, pos_tags


### 2.1 Dataset Statistics


In [ ]:
# Calculate dataset statistics
def calculate_dataset_stats(sentences, tags):
    num_sentences = len(sentences)
    num_words = sum(len(s) for s in sentences)
    avg_sentence_length = num_words / num_sentences if num_sentences > 0 else 0

    # Calculate vocabulary size
    vocab = set()
    for sentence in sentences:
        vocab.update(sentence)
    vocab_size = len(vocab)

    # Calculate tag distribution
    tag_counter = Counter()
    for tag_seq in tags:
        tag_counter.update(tag_seq)

    return {
        "num_sentences": num_sentences,
        "num_words": num_words,
        "avg_sentence_length": avg_sentence_length,
        "vocab_size": vocab_size,
        "tag_distribution": tag_counter
    }


## 3. Feature Extraction

We'll use pre-trained word embeddings as features for our POS tagger.


In [ ]:
# Create vocabulary and tag mappings
def create_mappings(train_sentences, train_tags):
    # Create word-to-index mapping
    word_to_idx = {"<PAD>": 0, "<UNK>": 1}  # Special tokens
    for sentence in train_sentences:
        for word in sentence:
            if word not in word_to_idx:
                word_to_idx[word] = len(word_to_idx)

    # Create tag-to-index mapping
    tag_to_idx = {}
    for tag_seq in train_tags:
        for tag in tag_seq:
            if tag not in tag_to_idx:
                tag_to_idx[tag] = len(tag_to_idx)

    # Create index-to-tag mapping for later use
    idx_to_tag = {idx: tag for tag, idx in tag_to_idx.items()}

    return word_to_idx, tag_to_idx, idx_to_tag

# Create character mappings
def create_char_mappings(sentences):
    char_to_idx = {"<PAD>": 0, "<UNK>": 1}
    for sentence in sentences:
        for word in sentence:
            for char in word:
                if char not in char_to_idx:
                    char_to_idx[char] = len(char_to_idx)

    return char_to_idx

# Create embedding matrix
def create_embedding_matrix(word_to_idx, word_vectors, embedding_dim):
    embedding_matrix = np.zeros((len(word_to_idx), embedding_dim))
    for word, idx in word_to_idx.items():
        if word in word_vectors:
            embedding_matrix[idx] = word_vectors[word]
        elif word == "<PAD>":
            embedding_matrix[idx] = np.zeros(embedding_dim)  # Zero vector for padding
        else:  # <UNK> or words not in pre-trained embeddings
            embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

    return embedding_matrix

# Prepare data for Keras RNN model
def prepare_data_for_rnn(sentences, tags, word_to_idx, tag_to_idx, max_length):
    """
    Prepare data for Keras RNN model.

    Args:
        sentences: List of sentences, where each sentence is a list of words
        tags: List of tag sequences, where each tag sequence is a list of tags
        word_to_idx: Dictionary mapping words to indices
        tag_to_idx: Dictionary mapping tags to indices
        max_length: Maximum sequence length

    Returns:
        X: Padded word index sequences
        y: Padded tag index sequences
        lengths: Original sequence lengths
    """
    X = []
    y = []
    lengths = []

    for sentence, tag_seq in zip(sentences, tags):
        # Convert words to indices
        word_indices = [word_to_idx.get(w, word_to_idx["<UNK>"]) for w in sentence]
        tag_indices = [tag_to_idx[t] for t in tag_seq]

        # Store the original length
        lengths.append(len(word_indices))

        X.append(word_indices)
        y.append(tag_indices)

    # Pad sequences
    X_padded = pad_sequences(X, maxlen=max_length, padding='post', value=word_to_idx["<PAD>"])
    y_padded = pad_sequences(y, maxlen=max_length, padding='post', value=0)  # 0 as padding tag index

    # Convert to numpy arrays
    lengths = np.array(lengths)

    return X_padded, y_padded, lengths

# Prepare character-level data
def prepare_char_data(sentences, char_to_idx, max_seq_length, max_word_length):
    X_char = []

    for sentence in sentences:
        sent_chars = []
        for word in sentence:
            word_chars = [char_to_idx.get(char, char_to_idx["<UNK>"]) for char in word[:max_word_length]]
            # Pad word to max_word_length
            word_chars = word_chars + [char_to_idx["<PAD>"]] * (max_word_length - len(word_chars))
            sent_chars.append(word_chars)

        # Pad sentence to max_seq_length
        if len(sent_chars) < max_seq_length:
            padding = [[char_to_idx["<PAD>"]] * max_word_length] * (max_seq_length - len(sent_chars))
            sent_chars = sent_chars + padding
        else:
            sent_chars = sent_chars[:max_seq_length]

        X_char.append(sent_chars)

    return np.array(X_char, dtype='int32')


## 4. Baseline Model

We'll implement a baseline model that tags each word with the most frequent tag it had in the training data.
For words not seen in the training data, the baseline will use the most frequent tag overall.


In [ ]:
class BaselineTagger:
    def __init__(self):
        self.word_to_tag = {}
        self.most_common_tag = None

    def train(self, sentences, tags):
        # Count word-tag occurrences
        word_tag_counts = defaultdict(Counter)
        tag_counts = Counter()

        for sentence, tag_seq in zip(sentences, tags):
            for word, tag in zip(sentence, tag_seq):
                word_tag_counts[word][tag] += 1
                tag_counts[tag] += 1

        # Find the most frequent tag for each word
        for word, tag_counter in word_tag_counts.items():
            self.word_to_tag[word] = tag_counter.most_common(1)[0][0]

        # Find the most common tag overall
        self.most_common_tag = tag_counts.most_common(1)[0][0]

    def predict(self, sentences):
        predictions = []

        for sentence in sentences:
            sentence_preds = []
            for word in sentence:
                # Use the most frequent tag for this word, or the most common tag if the word is unseen
                tag = self.word_to_tag.get(word, self.most_common_tag)
                sentence_preds.append(tag)
            predictions.append(sentence_preds)

        return predictions

# Evaluate predictions (original function)
def evaluate_predictions(true_tags, pred_tags, tag_to_idx):
    # Flatten the lists of tags
    true_flat = [tag for sent in true_tags for tag in sent]
    pred_flat = [tag for sent in pred_tags for tag in sent]

    # Ensure the flattened lists have the same length
    min_len = min(len(true_flat), len(pred_flat))
    true_flat = true_flat[:min_len]
    pred_flat = pred_flat[:min_len]

    # Convert tags to indices
    true_indices = [tag_to_idx[tag] for tag in true_flat]
    pred_indices = [tag_to_idx[tag] for tag in pred_flat]

    # Calculate metrics for each class
    precision = precision_score(true_indices, pred_indices, average=None, zero_division=0)
    recall = recall_score(true_indices, pred_indices, average=None, zero_division=0)
    f1 = f1_score(true_indices, pred_indices, average=None, zero_division=0)

    # Calculate macro-averaged metrics
    macro_precision = precision_score(true_indices, pred_indices, average='macro', zero_division=0)
    macro_recall = recall_score(true_indices, pred_indices, average='macro', zero_division=0)
    macro_f1 = f1_score(true_indices, pred_indices, average='macro', zero_division=0)

    # Calculate PR AUC for each class
    pr_auc = []
    for i in range(len(tag_to_idx)):
        # Convert to binary classification problem
        true_binary = [1 if t == i else 0 for t in true_indices]
        pred_binary = [1 if p == i else 0 for p in pred_indices]

        # Calculate precision-recall curve
        precision_curve, recall_curve, _ = precision_recall_curve(true_binary, pred_binary)
        pr_auc.append(auc(recall_curve, precision_curve))

    # Calculate macro-averaged PR AUC
    macro_pr_auc = np.mean(pr_auc)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "pr_auc": pr_auc,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "macro_pr_auc": macro_pr_auc
    }

# Function to evaluate predictions for sequence labeling
def evaluate_sequence_predictions(true_tags, pred_tags, tag_to_idx, max_length):
    # Flatten the predictions, excluding padding
    true_flat = []
    pred_flat = []

    for i, (true_seq, pred_seq) in enumerate(zip(true_tags, pred_tags)):
        for j, (true_tag, pred_tag) in enumerate(zip(true_seq, pred_seq)):
            # Skip padding tokens
            if true_tag != 0:  # 0 is padding
                true_flat.append(true_tag)
                pred_flat.append(pred_tag)

    # Calculate metrics
    precision = precision_score(true_flat, pred_flat, average=None, zero_division=0)
    recall = recall_score(true_flat, pred_flat, average=None, zero_division=0)
    f1 = f1_score(true_flat, pred_flat, average=None, zero_division=0)

    macro_precision = precision_score(true_flat, pred_flat, average='macro', zero_division=0)
    macro_recall = recall_score(true_flat, pred_flat, average='macro', zero_division=0)
    macro_f1 = f1_score(true_flat, pred_flat, average='macro', zero_division=0)

    # Calculate PR AUC for each class
    pr_auc = []
    for i in range(len(tag_to_idx)):
        true_binary = [1 if t == i else 0 for t in true_flat]
        pred_binary = [1 if p == i else 0 for p in pred_flat]

        if sum(true_binary) > 0:  # Only if class exists in true labels
            precision_curve, recall_curve, _ = precision_recall_curve(true_binary, pred_binary)
            pr_auc.append(auc(recall_curve, precision_curve))
        else:
            pr_auc.append(0.0)

    macro_pr_auc = np.mean(pr_auc)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "pr_auc": pr_auc,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "macro_pr_auc": macro_pr_auc
    }

# Function to convert baseline predictions to sequence format
def convert_baseline_to_sequence_format(sentences, predictions, tag_to_idx, max_length):
    y_baseline = []
    for sent, pred in zip(sentences, predictions):
        sent_indices = [tag_to_idx[tag] for tag in pred]
        # Pad to max_length
        sent_indices = sent_indices + [0] * (max_length - len(sent_indices))
        y_baseline.append(sent_indices[:max_length])
    return np.array(y_baseline)

# Plot training curves
def plot_training_curves(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Loss curves
    ax1.plot(history.history['loss'], label='Training Loss')
    ax1.plot(history.history['val_loss'], label='Validation Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.legend()
    ax1.grid(True)

    # Accuracy curves
    ax2.plot(history.history['accuracy'], label='Training Accuracy')
    ax2.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Training and Validation Accuracy')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()


## 5. RNN Model

Now we'll implement a Bidirectional Stacked RNN (with GRU or LSTM cells) for POS tagging.
We'll use the Functional API to build a model architecture that focuses on word embeddings.


In [ ]:
# Custom layers for masking
@register_keras_serializable()
class MaskNonPadding(Layer):
    def call(self, x):
        return tf.cast(tf.not_equal(x, 0), tf.float32)

    def get_config(self):
        return super().get_config()

@register_keras_serializable()
class ExpandMaskDim(Layer):
    def call(self, x):
        return tf.expand_dims(x, axis=-1)

    def get_config(self):
        return super().get_config()

def build_rnn_model(vocab_size, embedding_dim, embedding_matrix, max_length, n_tags,
                   rnn_units, dropout_rate, recurrent_dropout, rnn_type='LSTM',
                   use_char_embeddings=False, char_vocab_size=None, char_embedding_dim=None,
                   max_word_length=None):
    """
    Create a bidirectional stacked RNN model using Keras Functional API.

    Args:
        vocab_size: Size of the vocabulary
        embedding_dim: Dimension of word embeddings
        embedding_matrix: Pre-trained embedding matrix
        max_length: Maximum sequence length
        n_tags: Number of POS tags (output dimension)
        rnn_units: List of units for each RNN layer
        dropout_rate: Dropout probability
        recurrent_dropout: Recurrent dropout probability
        rnn_type: Type of RNN cell ('GRU' or 'LSTM')
        use_char_embeddings: Whether to use character-level embeddings
        char_vocab_size: Size of the character vocabulary
        char_embedding_dim: Dimension of character embeddings
        max_word_length: Maximum word length for character-level features

    Returns:
        A compiled Keras model
    """
    # Word input
    word_input = Input(shape=(max_length,), name='word_input')

    # Word embedding
    word_embedding = Embedding(
        vocab_size,
        embedding_dim,
        weights=[embedding_matrix],
        input_length=max_length,
        trainable=True,
        mask_zero=True,
        name='word_embedding'
    )(word_input)

    # Create mask from word input
    mask = MaskNonPadding()(word_input)
    mask = ExpandMaskDim()(mask)

    inputs = [word_input]
    embeddings = word_embedding

    # Add character-level embeddings if requested
    if use_char_embeddings and char_vocab_size is not None:
        char_input = Input(shape=(max_length, max_word_length), name='char_input')
        inputs.append(char_input)

        # Character embedding for each word
        char_embedding_layer = Embedding(char_vocab_size, char_embedding_dim,
                                       mask_zero=True, name='char_embedding')

        # Apply character embedding to each word
        char_embedded = TimeDistributed(char_embedding_layer)(char_input)

        # Encode each word's characters with bidirectional RNN
        if rnn_type == 'LSTM':
            char_encoder = TimeDistributed(
                Bidirectional(LSTM(char_embedding_dim // 2, return_sequences=False))
            )(char_embedded)
        else:
            char_encoder = TimeDistributed(
                Bidirectional(GRU(char_embedding_dim // 2, return_sequences=False))
            )(char_embedded)

        # Concatenate word and character embeddings
        embeddings = Concatenate()([word_embedding, char_encoder])

    # Apply mask to embeddings
    embeddings = embeddings * mask

    # Stack bidirectional RNN layers
    x = embeddings
    for i, units in enumerate(rnn_units):
        return_sequences = True  # Always return sequences for sequence labeling

        if rnn_type == 'LSTM':
            rnn_layer = Bidirectional(
                LSTM(units,
                     return_sequences=return_sequences,
                     dropout=dropout_rate,
                     recurrent_dropout=recurrent_dropout,
                     kernel_regularizer=l2(0.01))
            )
        else:
            rnn_layer = Bidirectional(
                GRU(units,
                    return_sequences=return_sequences,
                    dropout=dropout_rate,
                    recurrent_dropout=recurrent_dropout,
                    kernel_regularizer=l2(0.01))
            )

        x = rnn_layer(x)
        x = Dropout(dropout_rate)(x)

    # Output layer - one prediction per time step
    output = TimeDistributed(Dense(n_tags, activation='softmax'))(x)

    model = Model(inputs=inputs, outputs=output)
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


## 6. Training and Evaluation


In [ ]:
# This function has been replaced with direct model.fit() call in the training section

def predict_with_keras_model(model, inputs, lengths):
    """
    Make predictions with a trained Keras model.

    Args:
        model: Trained Keras model
        inputs: Input data (list of inputs)
        lengths: Original sequence lengths

    Returns:
        Predictions and probabilities
    """
    # Get model predictions
    y_pred_probs = model.predict(inputs)  # Shape: [batch_size, seq_len, num_classes]

    # Get the most likely tag for each word
    y_pred = np.argmax(y_pred_probs, axis=-1)  # Shape: [batch_size, seq_len]

    # Extract predictions up to the original sequence lengths
    all_preds = []
    all_probs = []

    for i, length in enumerate(lengths):
        all_preds.append(y_pred[i, :length])
        all_probs.append(y_pred_probs[i, :length])

    return all_preds, all_probs

def convert_predictions_to_tags(predictions, sentences, idx_to_tag):
    """
    Convert model predictions back to POS tags for each sentence.

    Args:
        predictions: List of arrays, where each array contains predicted tag indices for a sentence
        sentences: List of sentences (lists of words)
        idx_to_tag: Mapping from tag indices to tag names

    Returns:
        List of lists, where each inner list contains the predicted tags for a sentence
    """
    sentence_predictions = []

    for pred_indices, sentence in zip(predictions, sentences):
        # Convert indices to tags
        pred_tags = [idx_to_tag[idx] for idx in pred_indices[:len(sentence)]]
        sentence_predictions.append(pred_tags)

    return sentence_predictions


## 7. Main Execution


In [ ]:
# Download and load the data
print("Downloading and loading the data...")
treebank_dir = download_treebank(TREEBANK_URL, TREEBANK)
train_data, dev_data, test_data = load_conllu_data(treebank_dir, TREEBANK)

# Extract sentences and tags
print("Extracting sentences and tags...")
train_sentences, train_pos_tags = extract_sentences_and_tags(train_data)
dev_sentences, dev_pos_tags = extract_sentences_and_tags(dev_data)
test_sentences, test_pos_tags = extract_sentences_and_tags(test_data)


In [ ]:
# Calculate and display dataset statistics
print("Calculating dataset statistics...")
train_stats = calculate_dataset_stats(train_sentences, train_pos_tags)
dev_stats = calculate_dataset_stats(dev_sentences, dev_pos_tags)
test_stats = calculate_dataset_stats(test_sentences, test_pos_tags)

print(f"Dataset Statistics for {LANGUAGE} ({TREEBANK}):\n")
print("Training Set:")
print(f"  Number of sentences: {train_stats['num_sentences']}")
print(f"  Number of words: {train_stats['num_words']}")
print(f"  Average sentence length: {train_stats['avg_sentence_length']:.2f}")
print(f"  Vocabulary size: {train_stats['vocab_size']}")
print("\nDevelopment Set:")
print(f"  Number of sentences: {dev_stats['num_sentences']}")
print(f"  Number of words: {dev_stats['num_words']}")
print(f"  Average sentence length: {dev_stats['avg_sentence_length']:.2f}")
print("\nTest Set:")
print(f"  Number of sentences: {test_stats['num_sentences']}")
print(f"  Number of words: {test_stats['num_words']}")
print(f"  Average sentence length: {test_stats['avg_sentence_length']:.2f}")

# Display tag distribution
print("\nPOS Tag Distribution (Training Set):")
for tag, count in sorted(train_stats['tag_distribution'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {tag}: {count} ({count/train_stats['num_words']*100:.2f}%)")


In [ ]:
# Load pre-trained word embeddings
print("\nLoading pre-trained word embeddings...")
word_vectors = api.load("glove-wiki-gigaword-100")  # 100-dimensional GloVe embeddings
EMBEDDING_DIM = word_vectors.vector_size
print(f"Loaded {len(word_vectors.key_to_index)} word vectors with dimension {EMBEDDING_DIM}")

# Create mappings
print("Creating word and tag mappings...")
word_to_idx, tag_to_idx, idx_to_tag = create_mappings(train_sentences, train_pos_tags)
print(f"Vocabulary size: {len(word_to_idx)}")
print(f"Number of POS tags: {len(tag_to_idx)}")
print(f"POS tags: {list(tag_to_idx.keys())}")

# Create embedding matrix
print("Creating embedding matrix...")
embedding_matrix = create_embedding_matrix(word_to_idx, word_vectors, EMBEDDING_DIM)
print(f"Embedding matrix shape: {embedding_matrix.shape}")


In [ ]:
# Prepare data for Keras
print("Preparing data for Keras...")

# Prepare word-level data
X_train, y_train, train_lengths = prepare_data_for_rnn(train_sentences, train_pos_tags, word_to_idx, tag_to_idx, MAX_SEQUENCE_LENGTH)
X_dev, y_dev, dev_lengths = prepare_data_for_rnn(dev_sentences, dev_pos_tags, word_to_idx, tag_to_idx, MAX_SEQUENCE_LENGTH)
X_test, y_test, test_lengths = prepare_data_for_rnn(test_sentences, test_pos_tags, word_to_idx, tag_to_idx, MAX_SEQUENCE_LENGTH)

# Initialize input lists
train_inputs = [X_train]
dev_inputs = [X_dev]
test_inputs = [X_test]

# Create character mappings (needed for model building even if not used)
print("Creating character mappings...")
char_to_idx = create_char_mappings(train_sentences)
print(f"Character vocabulary size: {len(char_to_idx)}")

# Add character-level data processing if requested
if USE_CHAR_EMBEDDINGS:
    print("Preparing character-level data...")
    X_train_char = prepare_char_data(train_sentences, char_to_idx, MAX_SEQUENCE_LENGTH, MAX_WORD_LENGTH)
    X_dev_char = prepare_char_data(dev_sentences, char_to_idx, MAX_SEQUENCE_LENGTH, MAX_WORD_LENGTH)
    X_test_char = prepare_char_data(test_sentences, char_to_idx, MAX_SEQUENCE_LENGTH, MAX_WORD_LENGTH)

    train_inputs.append(X_train_char)
    dev_inputs.append(X_dev_char)
    test_inputs.append(X_test_char)

print(f"Training samples: {len(X_train)}")
print(f"Development samples: {len(X_dev)}")
print(f"Test samples: {len(X_test)}")


In [ ]:
# Train and evaluate the baseline model
print("\nTraining and evaluating the baseline model...")
baseline = BaselineTagger()
baseline.train(train_sentences, train_pos_tags)

train_baseline_preds = baseline.predict(train_sentences)
dev_baseline_preds = baseline.predict(dev_sentences)
test_baseline_preds = baseline.predict(test_sentences)

train_baseline_metrics = evaluate_predictions(train_pos_tags, train_baseline_preds, tag_to_idx)
dev_baseline_metrics = evaluate_predictions(dev_pos_tags, dev_baseline_preds, tag_to_idx)
test_baseline_metrics = evaluate_predictions(test_pos_tags, test_baseline_preds, tag_to_idx)

print("Baseline Model Results:\n")
print("Training Set:")
print(f"  Macro-averaged Precision: {train_baseline_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {train_baseline_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {train_baseline_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {train_baseline_metrics['macro_pr_auc']:.4f}")

print("\nDevelopment Set:")
print(f"  Macro-averaged Precision: {dev_baseline_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {dev_baseline_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {dev_baseline_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {dev_baseline_metrics['macro_pr_auc']:.4f}")

print("\nTest Set:")
print(f"  Macro-averaged Precision: {test_baseline_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {test_baseline_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {test_baseline_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {test_baseline_metrics['macro_pr_auc']:.4f}")


In [ ]:
# # Uncomment code to load from summary if needed
# from google.colab import drive
# from tensorflow import keras
# drive.mount('/content/drive')

# # Path to your .keras model file in Google Drive
# model_path = '/content/drive/MyDrive/best_rnn_pos_tagger.keras'

# # Load the model
# model = keras.models.load_model(model_path)

# # Check the model summary
# model.summary()

In [ ]:
# Train and evaluate the RNN model
print("\nTraining and evaluating the RNN model...")

# Build the model
print(f"Building {RNN_TYPE} model...")
rnn_model = build_rnn_model(
    vocab_size=len(word_to_idx),
    embedding_dim=EMBEDDING_DIM,
    embedding_matrix=embedding_matrix,
    max_length=MAX_SEQUENCE_LENGTH,
    n_tags=len(tag_to_idx),
    rnn_units=RNN_UNITS,
    dropout_rate=DROPOUT_RATE,
    recurrent_dropout=RECURRENT_DROPOUT,
    rnn_type=RNN_TYPE,
    use_char_embeddings=USE_CHAR_EMBEDDINGS,
    char_vocab_size=len(char_to_idx) if USE_CHAR_EMBEDDINGS else None,
    char_embedding_dim=CHAR_EMBEDDING_DIM,
    max_word_length=MAX_WORD_LENGTH
)

# Print model summary
rnn_model.summary()

# Set up callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_rnn_pos_tagger.keras', monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

# Train the model
print("Training RNN model...")
history = rnn_model.fit(
    train_inputs,
    y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(dev_inputs, y_dev),
    callbacks=callbacks,
    verbose=1
)


# Plot training curves
print("\nPlotting training curves...")
plot_training_curves(history)

# Extract training and validation losses from history for the existing loss curve plot
train_losses = history.history['loss']
dev_losses = history.history['val_loss']


In [ ]:
# Make predictions
print("Making predictions...")

# RNN predictions
train_rnn_pred = rnn_model.predict(train_inputs)
dev_rnn_pred = rnn_model.predict(dev_inputs)
test_rnn_pred = rnn_model.predict(test_inputs)

# Convert to class predictions
train_preds = np.argmax(train_rnn_pred, axis=-1)
dev_preds = np.argmax(dev_rnn_pred, axis=-1)
test_preds = np.argmax(test_rnn_pred, axis=-1)

# Store probabilities for later use
train_probs = train_rnn_pred
dev_probs = dev_rnn_pred
test_probs = test_rnn_pred

# Convert predictions to tags
train_pred_tags = convert_predictions_to_tags(train_preds, train_sentences, idx_to_tag)
dev_pred_tags = convert_predictions_to_tags(dev_preds, dev_sentences, idx_to_tag)
test_pred_tags = convert_predictions_to_tags(test_preds, test_sentences, idx_to_tag)

# Convert baseline predictions to sequence format
train_baseline_seq = convert_baseline_to_sequence_format(train_sentences, train_baseline_preds, tag_to_idx, MAX_SEQUENCE_LENGTH)
dev_baseline_seq = convert_baseline_to_sequence_format(dev_sentences, dev_baseline_preds, tag_to_idx, MAX_SEQUENCE_LENGTH)
test_baseline_seq = convert_baseline_to_sequence_format(test_sentences, test_baseline_preds, tag_to_idx, MAX_SEQUENCE_LENGTH)

# Evaluate the RNN model using sequence predictions
train_rnn_metrics = evaluate_sequence_predictions(y_train, train_preds, tag_to_idx, MAX_SEQUENCE_LENGTH)
dev_rnn_metrics = evaluate_sequence_predictions(y_dev, dev_preds, tag_to_idx, MAX_SEQUENCE_LENGTH)
test_rnn_metrics = evaluate_sequence_predictions(y_test, test_preds, tag_to_idx, MAX_SEQUENCE_LENGTH)

# Evaluate baseline using sequence predictions
train_baseline_metrics = evaluate_sequence_predictions(y_train, train_baseline_seq, tag_to_idx, MAX_SEQUENCE_LENGTH)
dev_baseline_metrics = evaluate_sequence_predictions(y_dev, dev_baseline_seq, tag_to_idx, MAX_SEQUENCE_LENGTH)
test_baseline_metrics = evaluate_sequence_predictions(y_test, test_baseline_seq, tag_to_idx, MAX_SEQUENCE_LENGTH)

print("RNN Model Results:\n")

# Print per-class metrics for the RNN model (Training Set)
print("\nPer-class Metrics for RNN Model (Training Set):")
print("Class\t\tPrecision\tRecall\t\tF1 Score")
print("-" * 60)
for i in range(len(tag_to_idx)):
    tag = idx_to_tag[i]
    precision = train_rnn_metrics['precision'][i]
    recall = train_rnn_metrics['recall'][i]
    f1 = train_rnn_metrics['f1'][i]
    pr_auc = train_rnn_metrics['pr_auc'][i]
    # Add padding to align columns based on tag length
    padding = "\t\t" if len(tag) < 8 else "\t"
    print(f"{tag}{padding}{precision:.4f}\t\t{recall:.4f}\t\t{f1:.4f}")

# Print per-class metrics for the RNN model (Development Set)
print("\nPer-class Metrics for RNN Model (Development Set):")
print("Class\t\tPrecision\tRecall\t\tF1 Score")
print("-" * 60)
for i in range(len(tag_to_idx)):
    tag = idx_to_tag[i]
    precision = dev_rnn_metrics['precision'][i]
    recall = dev_rnn_metrics['recall'][i]
    f1 = dev_rnn_metrics['f1'][i]
    pr_auc = dev_rnn_metrics['pr_auc'][i]
    # Add padding to align columns based on tag length
    padding = "\t\t" if len(tag) < 8 else "\t"
    print(f"{tag}{padding}{precision:.4f}\t\t{recall:.4f}\t\t{f1:.4f}")

# Print per-class metrics for the RNN model
print("\nPer-class Metrics for RNN Model (Test Set):")
print("Class\t\tPrecision\tRecall\t\tF1 Score")
print("-" * 60)
for i in range(len(tag_to_idx)):
    tag = idx_to_tag[i]
    precision = test_rnn_metrics['precision'][i]
    recall = test_rnn_metrics['recall'][i]
    f1 = test_rnn_metrics['f1'][i]
    pr_auc = test_rnn_metrics['pr_auc'][i]
    # Add padding to align columns based on tag length
    padding = "\t\t" if len(tag) < 8 else "\t"
    print(f"{tag}{padding}{precision:.4f}\t\t{recall:.4f}\t\t{f1:.4f}")

print("Training Set:")
print(f"  Macro-averaged Precision: {train_rnn_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {train_rnn_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {train_rnn_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {train_rnn_metrics['macro_pr_auc']:.4f}")

print("\nDevelopment Set:")
print(f"  Macro-averaged Precision: {dev_rnn_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {dev_rnn_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {dev_rnn_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {dev_rnn_metrics['macro_pr_auc']:.4f}")

print("\nTest Set:")
print(f"  Macro-averaged Precision: {test_rnn_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {test_rnn_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {test_rnn_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {test_rnn_metrics['macro_pr_auc']:.4f}")


In [ ]:
# Load MLP results from Assignment 3 for comparison
# We just copy pasted the results
mlp_train_metrics = {
    "macro_precision": 0.8789,
    "macro_recall": 0.8636,
    "macro_f1": 0.8702,
    "macro_pr_auc": 0.9023
}

mlp_dev_metrics = {
    "macro_precision": 0.8265,
    "macro_recall": 0.7940,
    "macro_f1": 0.8030,
    "macro_pr_auc": 0.8430
}

mlp_test_metrics = {
    "macro_precision": 0.8335,
    "macro_recall": 0.8091,
    "macro_f1": 0.8144,
    "macro_pr_auc": 0.8540
}

print("\nMLP Model Results (from Assignment 3):\n")
print("Training Set:")
print(f"  Macro-averaged Precision: {mlp_train_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {mlp_train_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {mlp_train_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {mlp_train_metrics['macro_pr_auc']:.4f}")

print("\nDevelopment Set:")
print(f"  Macro-averaged Precision: {mlp_dev_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {mlp_dev_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {mlp_dev_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {mlp_dev_metrics['macro_pr_auc']:.4f}")

print("\nTest Set:")
print(f"  Macro-averaged Precision: {mlp_test_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {mlp_test_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {mlp_test_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {mlp_test_metrics['macro_pr_auc']:.4f}")


In [ ]:
# Plot the loss curves
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss')
plt.plot(dev_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.savefig('rnn_loss_curves.png')
plt.show()


In [ ]:
# Plot per-class metrics for the RNN model
plt.figure(figsize=(12, 8))

# Get the tag names
tags = list(tag_to_idx.keys())

# Plot precision for each class
plt.subplot(2, 2, 1)
plt.bar(tags, test_rnn_metrics['precision'])
plt.title('Precision by POS Tag (Test Set)')
plt.xticks(rotation=90)
plt.ylim(0, 1)

# Plot recall for each class
plt.subplot(2, 2, 2)
plt.bar(tags, test_rnn_metrics['recall'])
plt.title('Recall by POS Tag (Test Set)')
plt.xticks(rotation=90)
plt.ylim(0, 1)

# Plot F1 for each class
plt.subplot(2, 2, 3)
plt.bar(tags, test_rnn_metrics['f1'])
plt.title('F1 Score by POS Tag (Test Set)')
plt.xticks(rotation=90)
plt.ylim(0, 1)

# Plot PR AUC for each class
plt.subplot(2, 2, 4)
plt.bar(tags, test_rnn_metrics['pr_auc'])
plt.title('PR AUC by POS Tag (Test Set)')
plt.xticks(rotation=90)
plt.ylim(0, 1)

plt.tight_layout()
plt.savefig('rnn_per_class_metrics.png')
plt.show()

# Plot confusion matrix for test set
print("Plotting confusion matrix...")

# Flatten the true and predicted tags for test set
true_tags_flat = []
pred_tags_flat = []
for i, (sentence, true_tags, pred_tags) in enumerate(zip(test_sentences, test_pos_tags, test_pred_tags)):
    for j, (word, true_tag, pred_tag) in enumerate(zip(sentence, true_tags, pred_tags)):
        true_tags_flat.append(tag_to_idx[true_tag])
        pred_tags_flat.append(tag_to_idx[pred_tag])

# Create confusion matrix
cm = confusion_matrix(true_tags_flat, pred_tags_flat)

# Normalize the confusion matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=[idx_to_tag[i] for i in range(len(tag_to_idx))],
            yticklabels=[idx_to_tag[i] for i in range(len(tag_to_idx))])
plt.title('Normalized Confusion Matrix')
plt.xlabel('Predicted Tags')
plt.ylabel('True Tags')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()

print("\nConfusion matrix saved as confusion_matrix.png")


In [ ]:
# Compare baseline, MLP, and RNN models
plt.figure(figsize=(10, 8))

# Prepare data for comparison
models = ['Baseline', 'MLP', 'RNN']
train_precision = [train_baseline_metrics['macro_precision'], mlp_train_metrics['macro_precision'], train_rnn_metrics['macro_precision']]
train_recall = [train_baseline_metrics['macro_recall'], mlp_train_metrics['macro_recall'], train_rnn_metrics['macro_recall']]
train_f1 = [train_baseline_metrics['macro_f1'], mlp_train_metrics['macro_f1'], train_rnn_metrics['macro_f1']]

dev_precision = [dev_baseline_metrics['macro_precision'], mlp_dev_metrics['macro_precision'], dev_rnn_metrics['macro_precision']]
dev_recall = [dev_baseline_metrics['macro_recall'], mlp_dev_metrics['macro_recall'], dev_rnn_metrics['macro_recall']]
dev_f1 = [dev_baseline_metrics['macro_f1'], mlp_dev_metrics['macro_f1'], dev_rnn_metrics['macro_f1']]

test_precision = [test_baseline_metrics['macro_precision'], mlp_test_metrics['macro_precision'], test_rnn_metrics['macro_precision']]
test_recall = [test_baseline_metrics['macro_recall'], mlp_test_metrics['macro_recall'], test_rnn_metrics['macro_recall']]
test_f1 = [test_baseline_metrics['macro_f1'], mlp_test_metrics['macro_f1'], test_rnn_metrics['macro_f1']]

# Plot comparison
x = np.arange(len(models))
width = 0.25

plt.subplot(3, 1, 1)
plt.bar(x - width, train_precision, width, label='Precision')
plt.bar(x, train_recall, width, label='Recall')
plt.bar(x + width, train_f1, width, label='F1')
plt.ylabel('Score')
plt.title('Training Set')
plt.xticks(x, models)
plt.legend()
plt.ylim(0, 1)

plt.subplot(3, 1, 2)
plt.bar(x - width, dev_precision, width, label='Precision')
plt.bar(x, dev_recall, width, label='Recall')
plt.bar(x + width, dev_f1, width, label='F1')
plt.ylabel('Score')
plt.title('Development Set')
plt.xticks(x, models)
plt.legend()
plt.ylim(0, 1)

plt.subplot(3, 1, 3)
plt.bar(x - width, test_precision, width, label='Precision')
plt.bar(x, test_recall, width, label='Recall')
plt.bar(x + width, test_f1, width, label='F1')
plt.ylabel('Score')
plt.title('Test Set')
plt.xticks(x, models)
plt.legend()
plt.ylim(0, 1)

plt.tight_layout()
plt.savefig('model_comparison.png')
plt.show()

print("\nVisualization files saved:")
print("- rnn_loss_curves.png: Training and validation loss curves")
print("- rnn_per_class_metrics.png: Per-class metrics for the RNN model")
print("- model_comparison.png: Comparison of baseline, MLP, and RNN models")


In [ ]:
# Plot confusion matrix for test set
print("Plotting confusion matrix...")

# Flatten the true and predicted tags for test set
true_tags_flat = []
pred_tags_flat = []
for i, (sentence, true_tags, pred_tags) in enumerate(zip(test_sentences, test_pos_tags, test_pred_tags)):
    for j, (word, true_tag, pred_tag) in enumerate(zip(sentence, true_tags, pred_tags)):
        true_tags_flat.append(tag_to_idx[true_tag])
        pred_tags_flat.append(tag_to_idx[pred_tag])

# Create confusion matrix
cm = confusion_matrix(true_tags_flat, pred_tags_flat)

# Normalize the confusion matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=[idx_to_tag[i] for i in range(len(tag_to_idx))],
            yticklabels=[idx_to_tag[i] for i in range(len(tag_to_idx))])
plt.title('Normalized Confusion Matrix')
plt.xlabel('Predicted Tags')
plt.ylabel('True Tags')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()

print("\nConfusion matrix saved as confusion_matrix.png")


In [ ]:
# Function to print example predictions
def print_example(sentence, true_tags, baseline_pred, rnn_pred, idx_to_tag):
    print("\nSentence:", " ".join(sentence))
    print("\nTrue Tags:", " ".join(true_tags))
    print("Baseline :", " ".join(baseline_pred))
    print("RNN Pred :", " ".join([idx_to_tag[idx] for idx in rnn_pred[:len(sentence)]]))

# Example predictions
print("Example Predictions:")
print("="*60)

# Select some examples from test set
example_indices = np.random.choice(len(test_sentences), 5, replace=False)
for i in example_indices:
    sentence = test_sentences[i]
    true_tags = test_pos_tags[i]
    baseline_pred = test_baseline_preds[i]
    rnn_pred = test_preds[i]
    print_example(sentence, true_tags, baseline_pred, rnn_pred, idx_to_tag)